# Paper-aligned UMNN Wind-FICA Experiment

This notebook runs and traces the paper NF (`UMNN_M_1`) case: IEEE-24, 24 hours, 38 generators, 5 wind farms, 45% wind capacity, 200 FICA scenarios sampled from a 1,000-scenario training pool, and 5,000 out-of-sample scenarios.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as ppnw
from pandapower.pd2ppc import _pd2ppc
from IPython.display import Image, display

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / 'STDE-CDM', *CWD.parents]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file() and (p / 'src/stde_cdm').is_dir()), None)
if ROOT is None: raise RuntimeError('Cannot locate the STDE-CDM project root')
POOL_FILE = ROOT / 'data/generated/wind_UMNN_M_1_z0-1-2-3-4_d0_n6000.npz'
TRAIN_POOL_SIZE = 1000
TEST_POOL_SIZE = 5000
N_WDR = 200
WIND_SHARE = 0.45
EPSILON = 0.03
THETA = 0.06
RUN_FICA = True  # Set False to reuse the newest result in the dedicated directory.
RUN_OUTPUT = ROOT / 'outputs/notebook_um_nn_aligned'
print('Project root:', ROOT)
print('Scenario pool:', POOL_FILE)

## 1. Load the five-farm NF scenario pool

In [ ]:
pool = np.load(POOL_FILE)
scenarios_pu = pool['scenarios_pu'].astype(float)
zones = pool['zones']
print('Shape:', scenarios_pu.shape, '= (scenario, hour, wind_farm)')
print('Zones:', zones.tolist())
print('Range:', scenarios_pu.min(), scenarios_pu.max())
assert scenarios_pu.shape == (6000, 24, 5)
assert np.all((scenarios_pu >= 0) & (scenarios_pu <= 1))

## 2. Split exactly as the aligned dispatch runner

The first 1,000 trajectories form the training pool. The remaining 5,000 are isolated for out-of-sample evaluation. The hourly point forecast for each farm is the median of the training pool only.

In [ ]:
train_scenarios = scenarios_pu[:TRAIN_POOL_SIZE]
test_scenarios = scenarios_pu[TRAIN_POOL_SIZE:TRAIN_POOL_SIZE + TEST_POOL_SIZE]
point_forecast = np.median(train_scenarios, axis=0)
train_errors = train_scenarios - point_forecast[None, :, :]
test_errors = test_scenarios - point_forecast[None, :, :]
print('Training pool:', train_scenarios.shape)
print('OOS test pool:', test_scenarios.shape)
print('Point forecast:', point_forecast.shape)
np.testing.assert_allclose(train_scenarios, point_forecast[None] + train_errors)
np.testing.assert_allclose(test_scenarios, point_forecast[None] + test_errors)
print('Identity verified: scenario = forecast + error')

In [ ]:
hours = np.arange(24)
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
for i in range(100):
    axes[0].plot(hours, train_scenarios[i].sum(axis=1), color='steelblue', alpha=.08)
axes[0].plot(hours, point_forecast.sum(axis=1), color='black', lw=2.5, label='Aggregate point forecast')
axes[0].set(title='Aggregate Five-Farm NF Wind Scenarios', ylabel='Aggregate power (p.u. sum)')
axes[0].legend(); axes[0].grid(alpha=.25)
for farm in range(5):
    axes[1].step(hours, point_forecast[:, farm], where='post', label=f'Wind farm {farm} (Zone {zones[farm]})')
axes[1].set(xlabel='Hour', ylabel='Power (p.u.)', title='Farm-Level Point Forecasts')
axes[1].set_xticks(hours); axes[1].legend(ncol=2); axes[1].grid(alpha=.25)
plt.tight_layout(); plt.show()

## 3. Apply the same 45% capacity scaling as the upstream five-solar-farm case

In [ ]:
net = ppnw.case24_ieee_rts()
pp.rundcpp(net)
_, ppci = _pd2ppc(net)
base_load_mw = float(ppci['bus'][:, 2].sum())
total_wind_capacity_mw = WIND_SHARE * base_load_mw
per_farm_capacity_mw = total_wind_capacity_mw / 5
WT_pred = point_forecast * per_farm_capacity_mw
WT_error_train = train_errors * per_farm_capacity_mw
WT_error_test = test_errors * per_farm_capacity_mw
print('IEEE-24 base load:', base_load_mw, 'MW')
print('Total wind capacity:', total_wind_capacity_mw, 'MW')
print('Capacity per wind farm:', per_farm_capacity_mw, 'MW')
print('WT_pred:', WT_pred.shape)
print('WT_error_train:', WT_error_train.shape)
print('WT_error_test:', WT_error_test.shape)

## 4. Run the aligned FICA case and verify its saved arrays

The runner loads the same UMNN pool, then FICA randomly selects 200 complete `(24 hours, 5 farms)` trajectories from the 1,000-scenario training pool. All 5,000 held-out trajectories are used only by `check_JCC`. This cell writes to a dedicated UMNN result directory, so an older AN result cannot be selected accidentally.

In [ ]:
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
if RUN_FICA:
    command = [
        sys.executable, '-m', 'stde_cdm.fica_system',
        '--scenario', str(POOL_FILE),
        '--train-pool-size', str(TRAIN_POOL_SIZE),
        '--test-pool-size', str(TEST_POOL_SIZE),
        '--network', 'case24_ieee_rts', '--method', 'FICA',
        '--T', '24', '--num-gen', '38', '--n-wdr', str(N_WDR),
        '--epsilon', str(EPSILON), '--theta', str(THETA),
        '--wind-share', str(WIND_SHARE), '--seed', '0',
        '--output', str(RUN_OUTPUT),
    ]
    run_env = os.environ.copy()
    previous_pythonpath = run_env.get('PYTHONPATH', '')
    run_env['PYTHONPATH'] = str(ROOT / 'src') + (os.pathsep + previous_pythonpath if previous_pythonpath else '')
    completed = subprocess.run(command, cwd=ROOT, env=run_env, check=True, text=True)

results = sorted(RUN_OUTPUT.glob('wind_fica_case24_ieee_rts_z0-1-2-3-4_d0_*.npz'))
assert results, 'No UMNN 200-scenario/5-farm FICA result found'
result_file = results[-1]
saved = np.load(result_file)
print('Loaded:', result_file.name)
np.testing.assert_allclose(saved['wind_pred'], WT_pred)
np.testing.assert_allclose(saved['train_errors'], WT_error_train)
np.testing.assert_allclose(saved['test_errors'], WT_error_test)
np.testing.assert_allclose(
    saved['gen_power'].sum(axis=1) + saved['wind_pred'].sum(axis=1),
    saved['load_bus_all'].sum(axis=1), atol=1e-5
)
summary = json.loads(result_file.with_suffix('.json').read_text(encoding='utf-8'))
assert Path(summary['scenario']).resolve() == POOL_FILE.resolve()
assert summary['n_wdr'] == N_WDR and summary['T'] == 24
print('All UMNN input arrays match the 200-scenario, 5-farm FICA run exactly')
print('OOS reliability:', summary['reliability'])
print('Day-ahead power balance holds for all 24 hours')

## 5. Paper-style visualization aligned with upstream `plot_paper`

In [ ]:
figure = RUN_OUTPUT / 'figures' / f'{result_file.stem}_paper.png'
assert figure.exists(), 'The matching UMNN paper-style figure was not generated'
print('Figure:', figure.name)
display(Image(filename=str(figure)))

### 5.1 Complete Figure 8 plotting code

The production implementation is maintained in `src/visualization.py`. The self-contained cell below repeats the complete plotting logic so that Figure 8 can be understood, modified, and regenerated directly from this notebook. It uses the saved FICA result and does not rerun the optimization.

In [ ]:
def plot_figure8_from_saved_result(saved_result, output_stem, test_errors, scenario_seed=10):
    """Reproduce the paper Figure 8 directly from one saved FICA result."""
    # Keep this local copy synchronized with src/visualization.py. These
    # settings are required; relying on the notebook kernel's previous
    # Matplotlib state makes two otherwise identical plots look different.
    plt.style.use('default')
    plt.rcParams.update({
        'mathtext.fontset': 'stix',
        'font.family': 'serif',
        'font.serif': ['STIXGeneral', 'Times New Roman', 'serif'],
        'font.size': 19,
        'legend.fontsize': 18,
        'xtick.labelsize': 19,
        'ytick.labelsize': 19,
        'axes.linewidth': 0.8,
        'axes.edgecolor': '#333333',
        'axes.titleweight': 'bold',
        'axes.labelweight': 'bold',
        'grid.color': '#C9C9C9',
        'grid.linewidth': 1.0,
        'grid.alpha': 0.3,
        'lines.linewidth': 3.0,
        'legend.frameon': True,
        'legend.framealpha': 0.92,
        'legend.edgecolor': '#CCCCCC',
        'legend.fancybox': True,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
    })
    gen_power = np.asarray(saved_result['gen_power'])
    gen_alpha = np.asarray(saved_result['gen_alpha'])
    gen_capacity = np.asarray(saved_result['gen_capacity'])
    gen_pmin = np.asarray(saved_result['gen_pmin'])
    gen_ramp = np.asarray(saved_result['gen_ramp'])
    gen_cost = np.asarray(saved_result['gen_cost'])
    wind_pred = np.asarray(saved_result['wind_pred'])
    test_errors = np.asarray(test_errors)

    horizon, num_gen = gen_power.shape
    if horizon < 2 or num_gen < 2:
        raise ValueError('Figure 8 requires at least 2 hours and 2 generators')

    title_kw = dict(fontsize=24, fontweight='bold', fontstyle='italic')
    label_kw = dict(fontsize=20, fontweight='bold', fontstyle='italic')
    legend_kw = dict(prop={
        'family': 'STIXGeneral', 'size': 18,
        'weight': 'bold', 'style': 'italic',
    })

    # Match the production routine exactly: show the two generators whose
    # participation factors vary most and one reproducible OOS realization.
    selected_generators = np.argsort(np.std(gen_alpha, axis=0))[-2:][::-1]
    scenario_index = int(np.random.RandomState(scenario_seed).choice(
        test_errors.shape[0], 1
    )[0])
    wind_error = test_errors.sum(axis=-1)[scenario_index]
    wind_forecast = wind_pred.sum(axis=-1)
    wind_actual = wind_forecast + wind_error

    num_rows = 1 + 3 * len(selected_generators)
    fig, axs = plt.subplots(
        num_rows, 1, figsize=(10, 2.52 * num_rows), squeeze=False
    )
    x = np.arange(horizon)

    ax = axs[0, 0]
    ax.step(x, wind_forecast, where='post', color='#1F77B4',
            linewidth=3.0, label='forecast')
    ax.step(x, wind_actual, where='post', color='#FF7F0E',
            linewidth=3.0, label='actual')
    ax.set_title('Wind Farm', **title_kw)
    ax.set_ylabel('Wind (MW)', **label_kw)
    ax.set_xlim(-0.5, horizon - 0.5)
    ax.set_xticks(np.arange(horizon))
    ax.legend(**legend_kw)
    ax.grid(True, linestyle='--', alpha=0.3, linewidth=1.0)

    for index, generator in enumerate(selected_generators):
        row = 1 + 3 * index
        actual_power = gen_power[:, generator] - gen_alpha[:, generator] * wind_error

        ax = axs[row, 0]
        ax.step(x, gen_power[:, generator], where='post', color='#1F77B4',
                linewidth=3.0, label='first-stage')
        ax.step(x, actual_power, where='post', color='#FF7F0E',
                linewidth=3.0, label='actual')
        ax.axhline(gen_pmin[generator], color='black', linestyle='--', linewidth=2.5)
        ax.axhline(gen_capacity[generator], color='black', linestyle='--', linewidth=2.5)
        ax.set_title(
            f'Gen {generator}, Cost {gen_cost[generator]:.2f} USD/MWh', **title_kw
        )
        ax.set_ylabel('Gen (MW)', **label_kw)
        ax.set_xlim(-0.5, horizon - 0.5)
        ax.set_xticks(np.arange(horizon))
        ax.legend(**legend_kw)
        ax.grid(True, linestyle='--', alpha=0.3, linewidth=1.0)

        ax = axs[row + 1, 0]
        ax.bar(x + 0.5, gen_alpha[:, generator], width=0.8,
               color='violet', label='AGC')
        ax.set_title(f'Gen {generator} AGC Factor', **title_kw)
        ax.set_ylabel('AGC Factor', **label_kw)
        ax.set_ylim(-1, 1)
        ax.set_xlim(-0.5, horizon - 0.5)
        ax.set_xticks(np.arange(horizon))
        ax.legend(**legend_kw)
        ax.grid(True, linestyle='--', alpha=0.5, linewidth=1.0)

        first_stage_ramp = np.diff(gen_power[:, generator])
        actual_ramp = np.diff(actual_power)
        ramp_x = np.arange(1, horizon) + 0.5
        bar_width = 0.35
        ramp_limit = gen_ramp[generator]

        ax = axs[row + 2, 0]
        ax.bar(ramp_x - bar_width / 2, first_stage_ramp, bar_width,
               color='steelblue', alpha=0.7, label='First-stage')
        ax.bar(ramp_x + bar_width / 2, actual_ramp, bar_width,
               color='darkorange', alpha=0.7, label='Actual')
        ax.axhline(ramp_limit, color='red', linestyle='--', linewidth=2.5,
                   label=f'Limit (±{ramp_limit:.1f} MW)')
        ax.axhline(-ramp_limit, color='red', linestyle='--', linewidth=2.5)
        ax.set_title(f'Gen {generator} Ramping', **title_kw)
        ax.set_ylabel('Ramp (MW/h)', **label_kw)
        ax.set_ylim(-1.2 * ramp_limit, 1.2 * ramp_limit)
        ax.set_xlim(-0.5, horizon - 0.5)
        ax.set_xticks(np.arange(horizon))
        ax.legend(loc='upper left', bbox_to_anchor=(0.0, 0.85), **legend_kw)
        ax.grid(True, linestyle='--', alpha=0.5, linewidth=1.0)

    for ax in axs.flat:
        ax.set_axisbelow(True)
        ax.tick_params(axis='both', which='major', labelsize=19, width=1.0)
        for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
            tick_label.set_fontweight('bold')

    axs[-1, 0].set_xlabel('Hour', **label_kw)
    fig.subplots_adjust(left=0.08, right=0.95, top=0.97, bottom=0.03, hspace=0.35)

    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    pdf_path = output_stem.with_suffix('.pdf')
    png_path = output_stem.with_suffix('.png')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    # The production routine uses a 300 dpi vector PDF and a lightweight
    # 180 dpi PNG preview. The PDF is the vector figure source.
    fig.savefig(png_path, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    return {
        'pdf': pdf_path,
        'png': png_path,
        'scenario_index': scenario_index,
        'selected_generators': selected_generators.tolist(),
    }


figure8_info = plot_figure8_from_saved_result(
    saved,
    RUN_OUTPUT / 'figures' / f'{result_file.stem}_figure8_notebook',
    saved['test_errors'],
)
print(figure8_info)

## 6. Interpretation

- `WT_pred` enters the deterministic day-ahead balance.
- Two hundred complete five-farm training-error trajectories sampled from the 1,000-scenario pool enter the FICA Wasserstein DRJCC approximation.
- Generator realization is `p[g,t] - alpha[g,t] * sum_w WT_error[s,t,w]`.
- Wind realization is `WT_pred[t,w] + WT_error[s,t,w]`.
- The 5,000 test trajectories never influence the solution; they measure out-of-sample joint reliability after the decision is fixed.

In [ ]:
# 7. Fig. 3 style dispatch trace from this notebook's in-memory FICA run
# This cell deliberately does not read the frozen paper day_00 backtest.
# The orange trace is one fixed OOS realization from this UMNN test pool,
# not a measured observation from the formal STDE-CDM real backtest.

current_result = saved
gen_power = np.asarray(current_result['gen_power'], dtype=float)
alpha = np.asarray(current_result['gen_alpha'], dtype=float)
wind_pred_mw = np.asarray(current_result['wind_pred'], dtype=float)
test_errors_mw = np.asarray(current_result['test_errors'], dtype=float)
gen_pmin = np.asarray(current_result['gen_pmin'], dtype=float)
gen_capacity = np.asarray(current_result['gen_capacity'], dtype=float)
gen_ramp = np.asarray(current_result['gen_ramp'], dtype=float)
gen_cost = np.asarray(current_result['gen_cost'], dtype=float)

# Verify that the plot consumes exactly the arrays produced above.
np.testing.assert_allclose(wind_pred_mw, WT_pred)
np.testing.assert_allclose(test_errors_mw, WT_error_test)
np.testing.assert_allclose(
    gen_power.sum(axis=1) + wind_pred_mw.sum(axis=1),
    np.asarray(current_result['load_bus_all']).sum(axis=1),
    atol=1e-5,
)

# Use the same reproducible OOS draw and generator selection rule as the
# notebook visualization above. Changing this seed changes only the
# displayed realization; it never changes the optimized FICA policy.
oos_seed = 10
oos_index = int(np.random.RandomState(oos_seed).choice(
    test_errors_mw.shape[0], 1
)[0])
wind_error = test_errors_mw[oos_index].sum(axis=1)
wind_forecast = wind_pred_mw.sum(axis=1)
wind_realization = wind_forecast + wind_error
realized_generation = gen_power - alpha * wind_error[:, None]
realized_ramp = np.diff(realized_generation, axis=0)
selected_generators = np.argsort(np.std(alpha, axis=0))[-2:][::-1]

plt.style.use('default')
plt.rcParams.update({
    'mathtext.fontset': 'stix',
    'font.family': 'serif',
    'font.serif': ['STIXGeneral', 'Times New Roman', 'serif'],
    'font.size': 19,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 19,
    'ytick.labelsize': 19,
    'legend.fontsize': 18,
    'axes.linewidth': 0.8,
    'axes.edgecolor': '#333333',
    'grid.color': '#C9C9C9',
    'grid.linewidth': 1.0,
    'lines.linewidth': 3.0,
    'legend.frameon': True,
    'legend.framealpha': 0.92,
    'legend.edgecolor': '#CCCCCC',
    'legend.fancybox': True,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

colors = {
    'forecast': '#1F77B4',
    'realization': '#FF7F0E',
    'first_stage': 'steelblue',
    'realized_dispatch': 'darkorange',
    'agc': 'violet',
    'limit': 'red',
}
title_kw = dict(fontsize=24, fontweight='bold', fontstyle='italic')
label_kw = dict(fontsize=24, fontweight='bold', fontstyle='italic')
legend_kw = dict(prop={
    'family': 'STIXGeneral', 'size': 18,
    'weight': 'bold', 'style': 'italic',
})

def style_fig3_axis(ax, grid_alpha):
    ax.set_axisbelow(True)
    ax.grid(True, linestyle='--', linewidth=1.0, alpha=grid_alpha)
    ax.tick_params(axis='both', which='major', labelsize=19, width=1.0)
    plt.setp(
        ax.get_xticklabels() + ax.get_yticklabels(),
        fontweight='bold', fontstyle='normal',
    )

hours = np.arange(gen_power.shape[0])
ramp_hours = np.arange(1, gen_power.shape[0])
fig, axes = plt.subplots(7, 1, figsize=(10, 17.64), squeeze=False)
axes = axes[:, 0]

ax = axes[0]
ax.step(hours, wind_forecast, where='post', color=colors['forecast'],
        label='forecast')
ax.step(hours, wind_realization, where='post', color=colors['realization'],
        label='OOS realization')
ax.set_title('Wind Farm', **title_kw)
ax.set_ylabel('Wind (MW)', **label_kw)
ax.legend(**legend_kw)
style_fig3_axis(ax, 0.3)

for position, generator in enumerate(selected_generators):
    base = 1 + 3 * position
    lower = float(gen_pmin[generator])
    upper = float(gen_capacity[generator])
    ramp_limit = float(gen_ramp[generator])

    ax = axes[base]
    ax.step(hours, gen_power[:, generator], where='post',
            color=colors['first_stage'], label='first stage')
    ax.step(hours, realized_generation[:, generator], where='post',
            color=colors['realized_dispatch'], label='OOS realization')
    ax.axhline(lower, color='black', linestyle='--', linewidth=2.5)
    ax.axhline(upper, color='black', linestyle='--', linewidth=2.5)
    ax.set_title(
        f'Gen {generator}, Cost {gen_cost[generator]:.2f} USD/MWh',
        **title_kw,
    )
    ax.set_ylabel('Gen (MW)', **label_kw)
    ax.legend(**legend_kw)
    style_fig3_axis(ax, 0.3)

    ax = axes[base + 1]
    ax.bar(hours + 0.5, alpha[:, generator], width=0.8,
           color=colors['agc'], label='AGC')
    ax.set_title(f'Gen {generator} AGC Factor', **title_kw)
    ax.set_ylabel('AGC Factor', **label_kw)
    ax.set_ylim(-1, 1)
    ax.legend(**legend_kw)
    style_fig3_axis(ax, 0.5)

    width = 0.35
    ax = axes[base + 2]
    ax.bar(ramp_hours - width / 2, np.diff(gen_power[:, generator]),
           width, color=colors['first_stage'], alpha=0.7,
           label='First stage')
    ax.bar(ramp_hours + width / 2, realized_ramp[:, generator],
           width, color=colors['realized_dispatch'], alpha=0.7,
           label='OOS realization')
    ax.axhline(ramp_limit, color=colors['limit'], linestyle='--',
               linewidth=2.5, label=f'Limit (±{ramp_limit:.1f} MW)')
    ax.axhline(-ramp_limit, color=colors['limit'], linestyle='--',
               linewidth=2.5)
    ax.set_ylim(-1.2 * ramp_limit, 1.2 * ramp_limit)
    ax.set_title(f'Gen {generator} Ramping', **title_kw)
    ax.set_ylabel('Ramp (MW/h)', **label_kw)
    ax.legend(loc='upper left', bbox_to_anchor=(0.0, 0.85), **legend_kw)
    style_fig3_axis(ax, 0.5)

for ax in axes:
    ax.set_xlim(-0.5, 23.5)
    ax.set_xticks(hours)
axes[-1].set_xlabel('Hour', **label_kw)

fig.subplots_adjust(
    left=0.08, right=0.95, top=0.97, bottom=0.03, hspace=0.35
)
fig3_stem = RUN_OUTPUT / 'figures' / f'{result_file.stem}_fig3_current_notebook'
fig3_pdf = fig3_stem.with_suffix('.pdf')
fig3_png = fig3_stem.with_suffix('.png')
fig.savefig(fig3_pdf, dpi=300, bbox_inches='tight')
fig.savefig(fig3_png, dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Result file:', result_file)
print('Data source: current UMNN notebook result; no frozen paper CASE used')
print('Displayed OOS scenario index:', oos_index)
print('Selected generators:', selected_generators.tolist())
print('PDF:', fig3_pdf)
print('PNG:', fig3_png)